In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib as mpl
import uuid
import os
import re
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error as mse
from scipy.stats import pearsonr
from sklearn.metrics import r2_score as r2
from pymatgen.core import Element, Structure
from pymatgen.io.vasp.outputs import Vasprun, Poscar
from tqdm.notebook import tqdm
from collections import OrderedDict, Counter
from scipy.spatial import ConvexHull
from scipy.constants import Boltzmann, elementary_charge
from zipfile import ZipFile
from joblib import Parallel, delayed

In [2]:
def encodeRGB(x):
    r, g, b = list(map(lambda x: max(0, min(int(x), 255)), re.split('[^\d]+', x)[1: 4]))
    return "#{0:02x}{1:02x}{2:02x}".format(r, g, b)
    

def formation_energy_per_atom(compound, neat_elements):
    fe = compound['relaxed_energy']
    formula = compound['formula']
    for element in formula.keys():
        element_energy = neat_elements.loc[element]['relaxed_energy_pa'].min()
        fe -= formula[element] * element_energy
    return fe / compound['natoms']


def element_content(element, data_point, relative=True):
    formula = data_point['formula']
    natoms = data_point['natoms']
    if element in formula.keys():
        return formula[element] / (natoms if relative else 1)
    return 0


def orderedDictFormula(structure, from_cif_mode=False):
    if from_cif_mode:
        structure = Structure.from_str(input_string=structure, fmt='cif')
    atomic_symbols = np.array(list(map(lambda x: x.symbol, structure.species)))    
    return OrderedDict(sorted(Counter(atomic_symbols).items()))

    
def formula2string(ordered_dict_formula, multiply=1):
    out_str=''
    ks = [k for k in ordered_dict_formula.keys() if (k != 'Np' and k != 'U')]
    for k in ks:
        out_str += f'{k}{ordered_dict_formula[k] * multiply}'
    return out_str


def getStructuresAsList(workdir):
    xml_list = [xml for xml in os.listdir(workdir) if xml.endswith('xml')]
    pkl_file_name = re.sub('3res_structures/', '3res_list.pkl', workdir)
    print('\ngetStructuresAsList out :', pkl_file_name)
    if os.path.exists(pkl_file_name):
        with open(pkl_file_name, 'rb') as f:
            result = pickle.load(f)
        if len(xml_list) == len(result):
            print(f'result file for {workdir} has been read from pkl')
            return result
    print(f'result file for {workdir} is being updated')
    result = []
    for xml_file in tqdm(xml_list):
        tag = re.sub('vasprun_', '', re.sub('.xml', '', xml_file))
        try:
            run = Vasprun(f'{workdir}/{xml_file}', parse_potcar_file=False)
        except:
            print(f'vasprun.xml is not OK for {tag}')
        if not run.converged:
            print(f'vasprun.xml IS NOT CONVERGED for {tag}')

        initial_structure = run.initial_structure
        final_structure = run.final_structure
        last_step_forces = np.array(run.ionic_steps[-1]['forces'])
        last_step_pressure = np.trace(run.ionic_steps[-1]['stress']) / 3
        energy = run.final_energy
        atomic_symbols = run.atomic_symbols
        atomic_numbers = np.array(list(map(lambda x: Element(x).number, atomic_symbols)))
        formula = OrderedDict(sorted(Counter(atomic_symbols).items()))
        natoms = len(atomic_symbols)
        nelements = len(formula)
        result += [[tag, formula, natoms, atomic_symbols, atomic_numbers, nelements, 
                    initial_structure, #initial_cell, initial_structure, 
                    final_structure, #final_cell, final_structure, 
                    last_step_pressure, last_step_forces, energy]]
    with open(pkl_file_name, 'wb') as f:
        pickle.dump(result, f)
    return result
    

def readVASPresults(data_group, data_subgroup='hs', return_neat_elements=False):
    RESULT = []
    for workdir in [f'dft/neat_elements/3res_structures/', 
                    f'dft/{data_group}_{data_subgroup}/3res_structures/']:
        result = getStructuresAsList(workdir)
        RESULT += result
    
    RESULT = pd.DataFrame(RESULT, 
                          columns=['compound', 'formula', 'natoms', 'atomic_symbols', 'atomic_numbers', 'nelements', 
                                   'initial_structure', #'cell', 'pos', 
                                   'relaxed_structure', #'relaxed_cell', 'relaxed_pos', 
                                   'relaxed_pressure', 'relaxed_forces', 'relaxed_energy']).sort_values('nelements').reset_index(drop=True)
    RESULT['relaxed_energy_pa'] = RESULT['relaxed_energy'] / RESULT['natoms']
    neat_elements = RESULT[RESULT['nelements'] == 1].copy()
    neat_elements['element'] = neat_elements['formula'].apply(lambda x: list(x.keys())[0])
    neat_elements.set_index('element', inplace=True)    
    RESULT = RESULT[RESULT['nelements'] > 1].copy()
    try:
        RESULT['index'] = RESULT['compound'].apply(lambda x: int(re.sub('.vasp', '', x)))
    except:
        RESULT['index'] = RESULT['compound'].apply(lambda x: re.sub('.vasp', '', x))
    RESULT.set_index('index', inplace=True, drop=True)
    RESULT['formation_energy_per_atom'] = [formation_energy_per_atom(RESULT.loc[i], neat_elements) for i in RESULT.index]

    print('datapoints of neat elements:', neat_elements.shape)
    print('datapoints w/o neat elements:', RESULT.shape)
    if return_neat_elements:
        return (RESULT, neat_elements)
    return RESULT

    
def simplexBasis(n):  # n == number of unique elements - 1 
    basis = np.vstack((np.eye(n), (1 + np.sqrt(n + 1)) / n * np.ones((1,n))))
    basis -= np.mean(basis, axis=0)
    return basis / np.sqrt(2)


def compositionTranform (composition, simplex_basis):
    composition = composition / np.sum(composition)  # norm composition to the element fractions
    return np.sum(composition.reshape(-1, 1) * simplex_basis, axis=0) 


def getEhull(point, hull, return_simplice=False):
    equations = hull.equations  
    simplices = hull.simplices
    # -2 index is for form.energies, -1 index contains biases, 
    mask = list(map(lambda x: all([hull.points[point_index][-1] < 1e-05 for point_index in x]), hull.simplices))
    ## the mask avoid consideration of simplices containing positive form.energy vertiсes
    equations = equations[mask] 
    simplices = simplices[mask]
    mask = (np.abs(equations[:,-2]) > 1e-05) & (np.abs(1 - equations[:, -2]) > 1e-05)
    ## all facets parallel to the energy axis or having zero slope are removed by the mask
    equations = equations[mask] 
    simplices = simplices[mask]
    planes = equations.T
    energy_coord = -(np.dot(planes[:-2].T, point) + planes[-1]) / planes[-2]
    ## among all possible intersection the correct one should possess the maximum energy
    maxen_index = np.argmax(energy_coord)
    if return_simplice:
        return energy_coord[maxen_index], simplices[maxen_index]
    return energy_coord[maxen_index]


def allegroInput(dataset_file):    
    print(f'Allegro input generation for {dataset_file}')
    data = pd.read_pickle(dataset_file)
    dataset_xyz = re.sub('pkl.gz', 'xyz', dataset_file)


def parseStructure(structure, from_cif_mode=False):
    if from_cif_mode:
        structure = Structure.from_str(cif_string, fmt='cif')
    lattice = structure.lattice.matrix
    positions = np.array([x.coords for x in structure])
    atomic_symbols = np.array(list(map(lambda x: x.symbol, structure.species)))
    return lattice, atomic_symbols, positions


def DefineItemPart(item_index, train_index, val_index, test_index):
    if item_index in train_index:
        return 'train'
    elif item_index in val_index:
        return 'val'
    elif item_index in test_index:
        return 'test'
    return ''
    

def DefineParts(result, space_sym_limit=8, random_state=0):
    inWhichPart = np.array(['train/val' if result.loc[i]['Space_group_no'] > space_sym_limit else 'test' 
                            for i in result.index])
    strat = result[inWhichPart == 'train/val']['Space_group_no'].values
    train, val = train_test_split(result[inWhichPart == 'train/val'], 
                                  test_size=0.1,
                                  random_state=random_state,
                                  stratify=strat)
    test = result[inWhichPart == 'test']
    inWhichPart = [DefineItemPart(i, train.index, val.index, test.index) for i in result.index]    
    return inWhichPart


def spaceGroupConventional(sg):
    sg = re.sub('-[\d]', lambda x: '\\bar{' + x.group()[1:] + '}', sg)
    return f'${sg}$'

    
def getDefectNumber(defect, def_type='substitution', at_from='S', at_to='Se'):
    defect_nsites = len(defect)
    subst = 0
    if def_type == 'substitution':
        for i in range(defect_nsites):
            if defect[i]['type'] == def_type and defect[i]['from'] == at_from and defect[i]['to'] == at_to:
                subst +=1
        return subst
    elif def_type == 'vacancy':
        for i in range(defect_nsites):
            if defect[i]['type'] == def_type and defect[i]['element'] == at_from:
                subst +=1
        return subst
    return -1


def getElementCount(item, element):
    if element in item['formula'].keys():
        return item['formula'][element]
    elif element == 'U':
        Me_vac =  getDefectNumber(item['defects'], def_type='vacancy', at_from='Mo')
        Me_vac += getDefectNumber(item['defects'], def_type='vacancy', at_from='W')
        return Me_vac
    elif element == 'Np' :
        nonMe_vac =  getDefectNumber(item['defects'], def_type='vacancy', at_from='S')
        nonMe_vac += getDefectNumber(item['defects'], def_type='vacancy', at_from='Se')
        return nonMe_vac
    return 0

    
elements = ['W', 'Mo', 'U', 'Se', 'S', 'Np']


def getFormula2Compare2DMD(item, elements=elements, multiplier=1):
    formula_str = ''
    for el in elements :
        el_number = int(multiplier * getElementCount(item, el))
        formula_str += f'{el}{el_number}'
    return formula_str

    
def getFormula2CompareFFCCS(item, elements=elements, multiplier=1):
    formula_str = ''
    for el in elements :
        el_number = int(multiplier * item[f'nsites_{el}'])
        formula_str += f'{el}{el_number}'
    return formula_str


kB = Boltzmann / elementary_charge
T_ = np.linspace(0, 2000, 41)

def Z(energies_eV_cell, weights, T):    
    '''partition function for group of the ccs entries (same composition and number of atoms)
       energies_eV_cell =  energies per cell (formation energies or relaxed energies)
       weights = weights of the structures from the ccs used (number of merged symmetrical realizations)
       T = temperature in Kelvin'''    
    if np.abs(T) < 1e-15:
        T = 1e-15
    Z = np.sum(weights * np.exp(-energies_eV_cell / (kB * T)))
    return Z
    

def T_delta_Sconf(energies_eV_cell, weights, T, natoms_for_per_atom=0):    
    '''configurational entropy contribution to free energies
       energies_eV_cell =  energies per cell (formation energies or relaxed energies)
       weights = weights of the structures from the ccs used (number of merged symmetrical realizations)
       T = temperature in Kelvin
       natoms_for_per_atom = number of atoms in the model cell from the ccs used'''    
    min_energy = energies_eV_cell.min()
    energies_eV_cell = energies_eV_cell - min_energy
    if np.abs(T) < 1e-15:
        T = 1e-15
    TdSconf = - T * kB * np.log(Z(energies_eV_cell, weights, T))
    if natoms_for_per_atom > 0:
        return (TdSconf + min_energy) / natoms_for_per_atom
    return TdSconf + min_energy

In [ ]:
neat_elements = pd.read_pickle("data/neat_elements_dataset.pkl.gz")
result = pd.read_pickle('data/ccs_dataset.pkl.gz')
unique_elements = ['W', 'Mo', 'Se', 'S']
simplex_basis = simplexBasis(len(unique_elements) - 1)
simplex_basis_df = pd.DataFrame(simplex_basis, columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)])
simplex_basis_df['formation_energy_pa'] = 0
simplex_basis_df['formula'] = neat_elements.loc[list(unique_elements)]['formula'].values
simplex_basis_df['formula_str'] = simplex_basis_df['formula'].apply(formula2string)
simplex_basis_df['compound'] = unique_elements
simplex_basis_df['part'] = 'Neat elements'
simplex_basis_df['size'] = 6

## DFT-derived structures
print('Collecting DFT-derived structures...')
compositions = []
for i in tqdm(result.index):
    compositions += [[element_content(element, result.loc[i], relative=False) for element in unique_elements]]
compositions = np.array(compositions)
simplex_coords_dft = pd.DataFrame(list(map(lambda x: compositionTranform(x, simplex_basis), compositions)),
                                  columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)])
simplex_coords_dft['formation_energy_pa'] = result['formation_energy_per_atom'].values
simplex_coords_dft['formula'] = result['formula'].values
simplex_coords_dft['formula_str'] = simplex_coords_dft['formula'].apply(formula2string)
simplex_coords_dft['compound'] = result['compound'].values
simplex_coords_dft['part'] = result['Space_group_no'].apply(lambda x: 'CCS (Pmm2 and higher)' if x > 8 else 'CCS (Cm and lower)').values
simplex_coords_dft['size'] = result['Space_group_no'].apply(lambda x: (4 if x > 8 else 1)).values

## 2DMD dataset
path_2dmd = 'data/2d-materials-point-defects-all-table.pkl.gz'
data_2dmd = pd.read_pickle(path_2dmd)
data_2dmd = data_2dmd[(data_2dmd['base'] == 'WSe2') | (data_2dmd['base'] == 'MoS2')].copy()
data_2dmd['natoms'] = data_2dmd['atomic_numbers'].apply(len)
data_2dmd['formation_energy_per_atom'] = data_2dmd['formation_energy'] / data_2dmd['natoms']
compositions = []
print('Collecting structures of the 2DMD dataset...')
for i in tqdm(data_2dmd.index):
    compositions += [[element_content(element, data_2dmd.loc[i], relative=False) for element in unique_elements]]
compositions = np.array(compositions)
simplex_coords_2dmd = pd.DataFrame(list(map(lambda x: compositionTranform(x, simplex_basis), compositions)),
                                   columns=[f'SC{i}' for i in range(len(simplex_basis) - 1)])
simplex_coords_2dmd['formation_energy_pa'] = data_2dmd['structure_formation_energy_per_atom'].values
simplex_coords_2dmd['formula'] = data_2dmd['formula'].values
simplex_coords_2dmd['formula_str'] = simplex_coords_2dmd['formula'].apply(formula2string)
simplex_coords_2dmd['compound'] = data_2dmd.index
simplex_coords_2dmd['part'] = '2DMD datasets'
simplex_coords_2dmd['size'] = .3

simplex_df = pd.concat((simplex_basis_df, simplex_coords_dft, simplex_coords_2dmd), axis=0)
simplex_df.set_index('compound', inplace=True, drop=True)

In [ ]:
fig = px.scatter_3d(simplex_df, x='SC0', y='SC1', z='SC2', color='part', size='size', 
                    color_discrete_sequence=px.colors.qualitative.Bold,
                    hover_data=[simplex_df['part'], 
                                simplex_df['formation_energy_pa'],
                                simplex_df.index, 
                                simplex_df['formula_str']])
fig.update_traces(marker=dict(line=dict(width=0,)), selector=dict(mode='markers'))

In [ ]:
camera = {"eye": {"x": 1.5, "y": 0.5, "z": 0.9}}
fig.update_layout(scene_camera=camera)
fig.show()

In [ ]:
available_energies_indexes = simplex_df.index
available_parts = simplex_df['part'].unique()
print(len(available_energies_indexes), 'energies available in ', ','.join(available_parts), 'parts')
points = simplex_df.iloc[:, :len(unique_elements)].values
hull = ConvexHull(points)
print(f'Convex hull dimensionality is {hull.ndim}D')

energies_df = pd.DataFrame(index=available_energies_indexes)
energies_df['Reference_hull_E'] = [getEhull(point, hull) for point in points[:, :3]]
energies_df['E_above_hull'] = simplex_df['formation_energy_pa'].values - energies_df['Reference_hull_E']
energies_df['E_above_hull'] = energies_df['E_above_hull'].apply(lambda x: x if np.abs(x) > 1e-05 else 0)

list_of_sets = list(map(set, hull.simplices))
list_of_hull_indexes = list(list_of_sets[0].union(*list_of_sets[1:]))
energies_df['DefineHull'] = np.isin(range(energies_df.shape[0]), list_of_hull_indexes) 
energies_df['DefineHull'] = (energies_df['DefineHull']) & \
                            (simplex_df['formation_energy_pa'].values < 1e-05) #negative energies only

energies_df = pd.concat((simplex_df, energies_df), axis=1)
energies_df['formula_str'] = energies_df['formula'].apply(formula2string)
energies_df['<25meV/atom_above'] = energies_df['E_above_hull'] < 25e-3
energies_df.info()

In [ ]:
result_energies_df = pd.merge(result, energies_df, how='left', left_on='compound', right_on=energies_df.index)
space_groups = dict(zip(result['Space_group_no'], result['Space_group_symbol']))
space_groups = OrderedDict(sorted(space_groups.items(), reverse=True))
sns.pairplot(result_energies_df[['W_nsites','U_nsites', 'Se_nsites', 'Np_nsites',
                                 'Space_group_symbol',
                                 'formation_energy_per_atom', 'E_above_hull', 
                                 'defect_nsites']], 
             hue='Space_group_symbol', corner=True,
             hue_order=[item[1] for item in space_groups.items() if item[0] in result['Space_group_no'].unique()], palette='crest')
plt.tight_layout()
plt.show()

In [ ]:
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=map(encodeRGB, px.colors.qualitative.Bold))
plt.figure(figsize=(6,4))
plt.hist([1000 * energies_df[energies_df['part'] == part]['E_above_hull'] for part in available_parts], 
         bins=np.linspace(0, 1600, 65), stacked=True, label=available_parts, edgecolor='grey')
plt.xlabel('$E_{hull}$, meV/atom')
plt.ylabel('Number of structures')
plt.legend()
plt.grid(alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()
plt.rcParams['axes.prop_cycle'] = plt.rcParamsDefault['axes.prop_cycle']

In [ ]:
missed_data = [col for col in energies_df.columns if not col in data_2dmd.columns]
data_2dmd_energies = pd.concat((data_2dmd, energies_df[energies_df['part'] == '2DMD datasets'][missed_data]), axis=1)
sns.pairplot(data_2dmd_energies[['defect_nsites', 'base', 'structure_formation_energy_per_atom', 'formation_energy_per_atom']], 
             height=3,
             hue='base', palette='tab10', corner=True, 
             plot_kws=dict(marker="o", linewidth=1, alpha=0.1))
plt.tight_layout()
plt.show()

print('Analysis of defect numbers remapping...')
print(data_2dmd['cell'].unique())
for dc in data_2dmd['defect_nsites'].unique():
    print(2 ** 2 * dc / 8 ** 2)

In [ ]:
sns.pairplot(data_2dmd_energies[['homo_lumo_gap', 'defect_nsites', 'formation_energy_per_atom', 'base', 
                                 'space_group_no', 'structure_formation_energy_per_atom', 'E_above_hull'
                                ]], 
             #height=3,
             hue='base', palette='tab10',corner=True, 
             plot_kws=dict(marker="o", linewidth=1, alpha=0.1)
            )
plt.tight_layout()
plt.savefig('figures/SI/2dmd_2bases_300dpi.png', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(9, 4))
numbers = data_2dmd.groupby(['defect_nsites','space_group_no']).size().to_frame()\
          .rename(mapper={0: 'Number_of_structures'}, axis=1).reset_index()\
          .pivot(columns='defect_nsites', index='space_group_no', values='Number_of_structures')
ccs_space_groups2add22dmd
numbers = pd.concat((numbers, pd.DataFrame(index=[sg for sg in ccs_space_groups2add22dmd 
                                                  if sg not in numbers.index])), axis=0)
numbers.sort_index(ascending=False, inplace=True)
numbers = numbers.T

ax = sns.heatmap(np.log10(numbers),
        cmap='YlOrBr',
        annot=numbers,
        linewidth=.1,
        cbar_kws={'label': 'lg (number of inequivalent structures)', 
                  'drawedges' : False, "shrink": 0.9},
        fmt='g',
        xticklabels=list(map(lambda x: f'{space_groups[x]}\n({x})', numbers.columns)),
        )
ax.set(xlabel="Space group", ylabel="Number of defect positions")
ax.xaxis.tick_top()
plt.savefig(f'figures/1_2dmd_symmetries.png', bbox_inches='tight', dpi=360)
plt.show()

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(8, 4), sharey=True)
for base in data_2dmd_energies['base'].unique():
    mask = data_2dmd_energies['base'] == base
    sns.scatterplot(x=1000 * data_2dmd_energies[mask]['formation_energy_per_atom'],
                    y=1000 * data_2dmd_energies[mask]['E_above_hull'], 
                    alpha=0.2, label=re.sub('2', '$_2$', base), ax=axs[1])
    sns.scatterplot(x=data_2dmd_energies[mask]['defect_nsites'], 
                    y=1000 * data_2dmd_energies[mask]['E_above_hull'], 
                    alpha=0.2, ax=axs[0])
axs[0].plot([3.5, 3.5], [0, 300], 'k', alpha=0.3, lw=1)
axs[0].text(3.5 - .5, 300, 'LDC', verticalalignment='top', horizontalalignment='right')
axs[0].text(3.5 + .5, 300, 'HDC', verticalalignment='top', horizontalalignment='left')
axs[1].plot([0, 300], [0, 300], 'k', alpha=0.3, lw=1, label='perfect\nmatch')
for i in range(2):
    axs[i].grid(alpha=0.3)
axs[1].legend(labelspacing=0.3)
axs[0].set_xlabel('Number of defect positions (2DMD)')
axs[1].set_xlabel('$E_{DF}$, meV/atom (2DMD)')
axs[0].set_ylabel('$E_{hull}$, meV/atom')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4.25, 4), sharey=True)
for base in data_2dmd_energies['base'].unique():
    mask = data_2dmd_energies['base'] == base
    sns.scatterplot(x=data_2dmd_energies[mask]['defect_nsites'], 
                    y=1000 * data_2dmd_energies[mask]['E_above_hull'], 
                    alpha=0.2, ax=ax, label=re.sub('2', '$_2$', base))
ax.plot([3.5, 3.5], [0, 300], 'k', alpha=0.3, lw=1)
ax.text(3.5 - .5, 300, 'LDC', verticalalignment='top', horizontalalignment='right')
ax.text(3.5 + .5, 300, 'HDC', verticalalignment='top', horizontalalignment='left')
ax.grid(alpha=0.3)
ax.legend(title='Base monolayer', labelspacing=0.3)
ax.set_xlabel('Number of defect positions (out of 192 available)')
ax.set_ylabel('2DMD datasets: $E_{hull}$, meV/atom')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(4.25, 4), sharey=True)
for base in data_2dmd_energies['base'].unique():
    mask = data_2dmd_energies['base'] == base
    sns.scatterplot(x=1000 * data_2dmd_energies[mask]['formation_energy_per_atom'],
                    y=1000 * data_2dmd_energies[mask]['E_above_hull'], 
                    alpha=0.2, label=re.sub('2', '$_2$', base), ax=ax)
ax.plot([0, 300], [0, 300], 'k', alpha=0.3, lw=1, label='perfect\nmatch')
ax.grid(alpha=0.3)
ax.legend(labelspacing=0.3)
ax.set_xlabel('$E_{DF}$, meV/atom (2DMD)')
ax.set_ylabel('$E_{hull}$, meV/atom')
plt.tight_layout()
plt.show()

In [ ]:
df_energies_available = energies_df.dropna().copy()
df_energies_available['size'] =  (3 - df_energies_available['E_above_hull'])
fig = px.scatter_3d(df_energies_available, x='SC0', y='SC1', z='SC2', color='E_above_hull', 
                    size='size', size_max=20, opacity=0.5,
                    hover_data=[df_energies_available['part'], df_energies_available.index, df_energies_available['formula_str']], 
                    color_continuous_scale='inferno')
fig.update_traces(marker=dict(line=dict(width=0)), selector=dict(mode='markers'))
fig.show()

In [ ]:
limit_Ehull = 25e-3
df_energies_on_hull = energies_df[(energies_df['E_above_hull'] < limit_Ehull) | (energies_df['part'] == '2DMD subsets')].copy()
df_energies_on_hull = df_energies_on_hull[df_energies_on_hull['E_above_hull'] < limit_Ehull]
df_energies_on_hull['size'] =  20 * (limit_Ehull - df_energies_on_hull['E_above_hull'])
fig = px.scatter_3d(df_energies_on_hull, x='SC0', y='SC1', z='SC2', color='E_above_hull',
                    size='size', size_max = 20, opacity=1,
                    hover_data=[df_energies_on_hull['part'], 
                                df_energies_on_hull['formation_energy_pa'],
                                df_energies_on_hull.index, 
                                df_energies_on_hull['formula_str']], 
                    color_continuous_scale='inferno')
fig.update_traces(marker=dict(line=dict(width=0)), selector=dict(mode='markers'))
fig.show()

In [ ]:
camera = dict(up=dict(x=-1, y=0, z=0),
              eye=dict(x=0, y=0, z=1.7)
             )
fig.update_layout(scene_camera=camera)
fig.show()